In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as funcy

In [12]:
class Deepthy(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1=nn.Sequential(nn.Conv2d(3,32,7,stride=2,padding=3),nn.ReLU())
        self.e2=nn.Sequential(nn.Conv2d(32,64,5,stride=2,padding=2),nn.ReLU())
        self.e3=nn.Sequential(nn.Conv2d(64,128,3,stride=2,padding=1),nn.ReLU())
        self.d1=nn.Sequential(nn.ConvTranspose2d(128,64,3,stride=2,padding=1,output_padding=1),nn.ReLU())
        self.d2=nn.Sequential(nn.ConvTranspose2d(128,32,3,stride=2,padding=1,output_padding=1),nn.ReLU())
        self.d3=nn.Sequential(nn.ConvTranspose2d(64,1,3,stride=2,padding=1,output_padding=1),nn.Sigmoid())
    def forward(self,x):
        ee1=self.e1(x)
        ee2=self.e2(ee1)
        ee3=self.e3(ee2)
        u1=self.d1(ee3)
        u2=self.d2(torch.cat((u1,ee2),dim=1))
        u3=self.d3(torch.cat((u2,ee1),dim=1))
        return u3

In [13]:
class Possy(nn.Module):
    def __init__(self):
        super().__init__()
        self.posy=nn.Sequential(nn.Conv2d(6,32,7,stride=2,padding=3),
                               nn.ReLU(),
                               nn.Conv2d(32,64,5,stride=2,padding=2),
                               nn.ReLU(),
                               nn.Conv2d(64,128,3,stride=2,padding=1),
                               nn.ReLU(),
                               nn.AdaptiveAvgPool2d(1),
                               nn.Flatten(),nn.Linear(128,6))
    def forward(self,i1,i2):
        x=torch.cat((i1,i2),dim=1)
        return self.posy(x)

In [4]:
def three3d(dep,kin):
    b,_,h,w=dep.shape
    dev=dep.device
    y,x=torch.meshgrid(torch.arange(h,device=dev),torch.arange(w,device=dev),
                      indexing='ij')
    noe=torch.ones_like(x)
    pixx=torch.stack([x,y,noe],dim=0).float()
    pixx=pixx.unsqueeze(0).repeat(b,1,1,1)
    rea=kin@pixx.view(b,3,-1)
    rea=rea.view(b,3,h,w)*dep
    return rea

In [15]:
def two2d(poi,k):
    b,_,h,w=poi.shape
    pt=poi.view(b,3,-1)
    pixx=k@pt
    pixx=pixx[:,:2]/(pixx[:,2:3]+1e-7)
    return pixx.view(b,2,h,w)

In [6]:
def warp(img,flow):
    b,c,h,w=img.shape
    gy,gx=torch.meshgrid(torch.linspace(-1,1,h,device=img.device),torch.linspace(-1,1,w,device=img.device),
                        indexing='ij')
    grid=torch.stack([gx,gy],dim=-1)
    grid=grid.unsqueeze(0).repeat(b,1,1,1)
    nflow=torch.zeros_like(flow)
    nflow[:,0]=flow[:,0]/(w/2)
    nflow[:,1]=flow[:,1]/(h/2)
    grid=grid+nflow.permute(0,2,3,1)
    return funcy.grid_sample(img,grid,align_corners=True)

In [14]:
class Floww(nn.Module):
    def __init__(self):
        super().__init__()
        self.fnet=nn.Sequential(nn.Conv2d(6,32,7,padding=3),
                               nn.ReLU(),nn.Conv2d(32,64,5,padding=2),
                               nn.ReLU(),nn.Conv2d(64,2,3,padding=1))
    def forward(self,it,it1):
        x=torch.cat([it,it1],dim=1)
        return self.fnet(x)

In [8]:
def mmask(rloss,nrloss,temp=0.1):
    sub=rloss-nrloss
    return torch.sigmoid(sub/temp)

In [16]:
def photoo(i,it):
    return (i-it).abs().mean(1,keepdim=True)

In [10]:
def ssim(x,y):
    c1,c2=0.01**2,0.03**2
    mux=funcy.avg_pool2d(x,3,1,1)
    muy=funcy.avg_pool2d(y,3,1,1)
    sx=funcy.avg_pool2d(x*x,3,1,1)-mux**2
    sy=funcy.avg_pool2d(y*y,3,1,1)-muy**2
    sxy=funcy.avg_pool2d(x*y,3,1,1)-mux*muy
    nssim=(2*mux*muy+c1)*(2*sxy+c2)
    dssim=(mux**2+muy**2+c1)*(sx+sy+c2)
    return torch.clamp((1-nssim/dssim)/2,0,1)

In [11]:
def smooth(dep,im):
    ddx=torch.abs(dep[:,:,:,1:]-dep[:,:,:,:-1])
    ddy=torch.abs(dep[:,:,1:,:]-dep[:,:,:-1,:])
    dxi=torch.mean(torch.abs(im[:,:,:,1:]-im[:,:,:,:-1]),1,keepdim=True)
    dyi=torch.mean(torch.abs(im[:,:,1:,:]-im[:,:,:-1,:]),1,keepdim=True)
    loss=(ddx*torch.exp(-dxi)).mean()+(ddy*torch.exp(-dyi)).mean()
    return loss